In [1]:
import barcode
from barcode.writer import ImageWriter

In [2]:
from reportlab.lib.pagesizes import mm
from reportlab.platypus import SimpleDocTemplate, Image, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.enums import TA_CENTER
from io import BytesIO
from reportlab.platypus import Image
from reportlab.platypus import PageBreak
import qrcode
import json


In [ ]:

# ========================
# DATI PRODOTTO
# ========================
nome = "Cuffie Bluetooth XYZ"
prezzo = "€29.99"
codice = "123456789012FA"

# ========================
# GENERA BARCODE
# ========================
CODE128 = barcode.get_barcode_class('code128')
barcode_img = CODE128(codice, writer=ImageWriter())
barcode_file = barcode_img.save('barcode_temp')

In [ ]:


# ========================
# CREA PDF
# ========================
doc = SimpleDocTemplate(
    "etichetta.pdf",
    pagesize=(60*mm, 40*mm), 
        leftMargin=2,
    rightMargin=2,
    topMargin=2,
    bottomMargin=2
)

styles = getSampleStyleSheet()

# Stile centrato
style_center = styles["Normal"]
style_center.alignment = TA_CENTER
style_center.fontSize = 7
elements = []

# Nome prodotto
elements.append(Paragraph(f"<b>{nome}</b>", style_center))
elements.append(Spacer(1, 1))

# Prezzo
elements.append(Paragraph(prezzo, style_center))
elements.append(Spacer(1, 1))

# Barcode
elements.append(Image(barcode_file, width=50*mm, height=12*mm))
elements.append(Spacer(1, 1))

# Codice sotto barcode
elements.append(Paragraph(codice, style_center))

# Costruisci PDF
doc.build(elements)

print("Etichetta generata: etichetta.pdf")

In [3]:
def genera_etichetta_qr(data: dict, buffer: bool = False, name: str = "etichetta.pdf"):

    qr = qrcode.QRCode(
        version=1,
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=3,
        border=1
    )

    #qr.add_data(data["codice"])
    label = ""
    for key, value in data.items():
        label += f"{value} | "
    qr.add_data(label)
    qr.make(fit=True)

    img = qr.make_image(fill_color="black", back_color="white")
    if buffer:  
        buffer = BytesIO()
        img.save(buffer)
        buffer.seek(0)
        return buffer
    else:
        img.save(name)
        return name
    
def genera_etichetta_barcode(data: dict, buffer: bool = False, name: str = "etichetta.pdf"):
    CODE128 = barcode.get_barcode_class('code128')
    barcode_img = CODE128(data["codice"], writer=ImageWriter())
    if buffer:
        buffer = BytesIO()
        barcode_img.write(buffer)
        buffer.seek(0)
        return buffer
    else:
        barcode_file = barcode_img.save(name)
        return barcode_file
    

In [4]:


elements = []
carte = [
    {"nome": "Pikachu V", "prezzo": "€6.50", "codice": "PK-SWSH-001"},
    {"nome": "Charizard VSTAR", "prezzo": "€95.00", "codice": "CR-SWSH-015"},
    {"nome": "Bulbasaur", "prezzo": "€3.20", "codice": "BS-BASE-044"},
    {"nome": "Squirtle", "prezzo": "€3.80", "codice": "SQ-BASE-007"},
    {"nome": "Jigglypuff", "prezzo": "€2.10", "codice": "JP-JUN-039"},
    {"nome": "Gengar EX", "prezzo": "€42.00", "codice": "GE-XY-094"},
    {"nome": "Mewtwo EX", "prezzo": "€88.00", "codice": "MW-XY-055"},
    {"nome": "Eevee", "prezzo": "€5.00", "codice": "EV-SWSH-101"},
    {"nome": "Dragonite V", "prezzo": "€35.00", "codice": "DR-SWSH-049"},
    {"nome": "Snorlax", "prezzo": "€18.00", "codice": "SN-BASE-143"},
]

doc = SimpleDocTemplate(
    "etichetta.pdf",
    pagesize=(60*mm, 25*mm), 
        leftMargin=2,
    rightMargin=2,
    topMargin=2,
    bottomMargin=2
)

styles = getSampleStyleSheet()

# Stile centrato
style_center = styles["Normal"]
style_center.alignment = TA_CENTER
style_center.fontSize = 7
#width_code, height_code = 50*mm, 15*mm
width_code, height_code = 12*mm, 12*mm

for i, carta in enumerate(carte):
    print(f"Generando etichetta per {carta['nome']}...")
    #code_buffer = genera_etichetta_barcode(carta, buffer=True)
    code_budder = genera_etichetta_qr(carta, buffer=True)


    elements.append(Paragraph(carta["nome"] + " - " + carta["prezzo"], style_center))
    elements.append(Image(code_budder, width=width_code, height=height_code))
    #elements.append(Paragraph(carta["codice"], style_center))

    if i != len(carte) - 1:
        elements.append(PageBreak())

doc.build(elements)

print("Etichetta generata: etichetta.pdf")

Generando etichetta per Pikachu V...
Generando etichetta per Charizard VSTAR...
Generando etichetta per Bulbasaur...
Generando etichetta per Squirtle...
Generando etichetta per Jigglypuff...
Generando etichetta per Gengar EX...
Generando etichetta per Mewtwo EX...
Generando etichetta per Eevee...
Generando etichetta per Dragonite V...
Generando etichetta per Snorlax...
Etichetta generata: etichetta.pdf
